In [9]:
#EXCERCISE 1 - Bernstein-Vaziriani algorithm

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorSampler
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeKyiv


# K = 0 1 0 0 1 0 1 1

def bv_query(s):
    # Create a quantum circuit implementing a query gate for the
    # Bernstein-Vazirani problem.

    qc = QuantumCircuit(len(s) + 1)
    for index, bit in enumerate(reversed(s)):
        if bit == "1":
            qc.cx(index, len(s))
    return qc


# display(bv_query("01001011").draw(output="mpl")) #ASCII for K like Kacper


def compile_circuit(function: QuantumCircuit):
    # Compiles a circuit for use in the Deutsch-Jozsa (or Bernstein-Vazirani) algorithm.

    n = function.num_qubits - 1
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.compose(function, inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


def bv_algorithm(function: QuantumCircuit):
    qc = compile_circuit(function)
    result = AerSimulator().run(qc, shots=1, memory=True).result()
    return result.get_memory()[0]


oracle = bv_query("01001011")
qc = compile_circuit(oracle)
display(bv_algorithm(qc)) #ASCII for K like Kacper

sampler = StatevectorSampler()
result = sampler.run([qc], shots=1024).result()
print(result[0].data)
print(result[0].data.keys())
counts = result[0].data.c.get_counts()

print("Noiseless:", counts)

backend = FakeKyiv()
qc_noisy = transpile(qc, backend)

sampler_noisy = SamplerV2(backend)
result_noisy = sampler_noisy.run([qc_noisy], shots=1024).result()
counts_noisy = result_noisy[0].data.c.get_counts()

print("Noisy:", counts_noisy)



'11100101'

DataBin(c=BitArray(<shape=(), num_shots=1024, num_bits=8>))
dict_keys(['c'])
Noiseless: {'01001011': 1024}
Noisy: {'01001011': 943, '01011011': 1, '01001001': 12, '00001011': 11, '00000001': 6, '01001010': 14, '00000011': 11, '01000011': 19, '01001111': 1, '01101011': 2, '00000000': 3, '00001001': 1}


In [ ]:
#Simon algorithm
import numpy as np

def simon_function(s: str):
    """
    Create a QuantumCircuit implementing a query gate for Simon problem obeying the promise for the hidden string `s`
    """
    # Our quantum circuit has 2n qubits for n = len(s)
    n = len(s)
    qc = QuantumCircuit(2 * n)

    # Define a random permutation of all n bit strings. This permutation will effectively hide the string s.
    pi = np.random.permutation(2**n)

    # Now we'll define a query gate explicitly. The idea is to first define a function g(x) = min{x,x ^ s}, which
    # is a simple function that satisfies the promise, and then we take f to be the composition of g and the random
    # permutation pi. This gives us a random function satisfying the promise for s.

    query_gate = np.zeros((4**n, 4**n))
    for x in range(2**n):
        for y in range(2**n):
            z = y ^ pi[min(x, x ^ int(s, 2))]
            query_gate[x + 2**n * z, x + 2**n * y] = 1

    # Our circuit has just this one query gate
    qc.unitary(query_gate, range(2 * n))
    return qc


    #library galois - useful to solve classical problem after simon

'01001011'